In [ ]:
import os
from google.colab import drive
import rasterio
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# Step 1. Extract China pixels from downloaded NH GEE CSV files
# Author: ChatGPT
# Purpose:
#   1) Read GEE-exported Northern Hemisphere static/dynamic CSVs
#   2) Filter China pixels by bbox and optional China boundary polygon
#   3) Save clean China-only dynamic/static parquet files
# ============================================================

import os
import re
import glob
import warnings
import pandas as pd
import numpy as np

warnings.filterwarnings("ignore")

# ----------------------------
# 0. User settings
# ----------------------------

# Folder containing downloaded GEE CSV files
DATA_DIR = "/content/drive/MyDrive/GEE_NH_0_65_VegAgri_Yearly"

# Output folder
OUT_DIR = "/content/drive/MyDrive/GEE_China_VegClimate_NO"
os.makedirs(OUT_DIR, exist_ok=True)

# Optional China boundary file.
# Recommended: GADM China ADM0 shapefile or geopackage.
# Examples:
# CHINA_BOUNDARY_PATH = "/content/drive/MyDrive/China_boundary/gadm41_CHN_0.shp"
# CHINA_BOUNDARY_PATH = "/content/drive/MyDrive/China_boundary/gadm41_CHN_0.gpkg"
# If not available, set to None and only bbox filter will be used.
CHINA_BOUNDARY_PATH = None

# China rough bounding box for fast pre-filter
CHINA_LON_MIN = 73.0
CHINA_LON_MAX = 135.5
CHINA_LAT_MIN = 18.0
CHINA_LAT_MAX = 54.5

# Target years
YEARS = list(range(2018, 2026))  # adjust if needed

# Core columns
DYNAMIC_KEEP_COLS = [
    "grid_id", "lon", "lat", "region_id",
    "year", "step16", "date_start", "date_end", "doy_start", "doy_end",
    "EVI", "LAI",
    "VPD", "LST_Day", "SM_rootproxy", "Precip", "Tair"
]

STATIC_KEEP_COLS = [
    "grid_id", "lon", "lat", "region_id",
    "year_static",
    "elevation", "slope", "LC_Type1",
    "lc_veg_frac", "evi_peak_frac", "evi_amp_frac", "valid_frac",
    "Greenup_1", "Peak_1", "Dormancy_1"
]


# ----------------------------
# 1. Helper functions
# ----------------------------

def list_files_for_year(data_dir, year, kind):
    """
    kind: 'dynamic' or 'static'
    Expected file names:
      NH0_65_dynamic_y2025_R_*.csv
      NH0_65_static_y2025_R_*.csv
    """
    pattern = os.path.join(data_dir, f"*{kind}*y{year}*.csv")
    files = sorted(glob.glob(pattern))
    return files


def bbox_filter(df):
    """Fast China bbox filter using lon/lat columns."""
    return df[
        (df["lon"] >= CHINA_LON_MIN) &
        (df["lon"] <= CHINA_LON_MAX) &
        (df["lat"] >= CHINA_LAT_MIN) &
        (df["lat"] <= CHINA_LAT_MAX)
    ].copy()


def load_china_boundary(boundary_path):
    """
    Load China boundary polygon if geopandas is available and file exists.
    """
    if boundary_path is None:
        return None

    if not os.path.exists(boundary_path):
        print(f"[WARN] China boundary file not found: {boundary_path}")
        return None

    try:
        import geopandas as gpd
        china = gpd.read_file(boundary_path)
        china = china.to_crs("EPSG:4326")
        # dissolve into one polygon
        china = china.dissolve()
        print("[OK] Loaded China boundary:", boundary_path)
        return china
    except Exception as e:
        print("[WARN] Failed to load China boundary. Fallback to bbox only.")
        print("       Error:", repr(e))
        return None


def polygon_filter_points(df, china_boundary):
    """
    Precise polygon filter using grid center points.
    Requires geopandas.
    """
    if china_boundary is None:
        return df

    import geopandas as gpd

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs="EPSG:4326"
    )

    # spatial join: keep points within China polygon
    gdf = gpd.sjoin(
        gdf,
        china_boundary[["geometry"]],
        how="inner",
        predicate="within"
    )

    gdf = gdf.drop(columns=[c for c in ["geometry", "index_right"] if c in gdf.columns])
    return pd.DataFrame(gdf)


def safe_keep_columns(df, keep_cols):
    """Keep available columns only and warn missing columns."""
    existing = [c for c in keep_cols if c in df.columns]
    missing = [c for c in keep_cols if c not in df.columns]
    if missing:
        print("[WARN] Missing columns:", missing)
    return df[existing].copy()


def read_csv_light(path, keep_cols=None):
    """
    Read CSV with selected columns if possible.
    If usecols fails due to missing columns, read full then subset.
    """
    try:
        if keep_cols is not None:
            return pd.read_csv(path, usecols=lambda c: c in keep_cols)
        else:
            return pd.read_csv(path)
    except Exception:
        df = pd.read_csv(path)
        if keep_cols is not None:
            df = safe_keep_columns(df, keep_cols)
        return df


# ----------------------------
# 2. Load optional China boundary
# ----------------------------

china_boundary = load_china_boundary(CHINA_BOUNDARY_PATH)


# ----------------------------
# 3. Extract dynamic data
# ----------------------------

all_dynamic_parts = []

for year in YEARS:
    files = list_files_for_year(DATA_DIR, year, "dynamic")
    print(f"\n[DYNAMIC] Year {year}: {len(files)} files")

    if len(files) == 0:
        continue

    year_parts = []

    for fp in files:
        try:
            df = read_csv_light(fp, DYNAMIC_KEEP_COLS)

            if "lon" not in df.columns or "lat" not in df.columns:
                print("[SKIP] No lon/lat:", os.path.basename(fp))
                continue

            df = bbox_filter(df)

            if len(df) == 0:
                continue

            df = safe_keep_columns(df, DYNAMIC_KEEP_COLS)
            year_parts.append(df)

        except Exception as e:
            print("[ERROR]", os.path.basename(fp), repr(e))

    if len(year_parts) == 0:
        print(f"[DYNAMIC] Year {year}: no China bbox rows")
        continue

    year_df = pd.concat(year_parts, ignore_index=True)

    # Polygon refine once per year to reduce cost
    year_df = polygon_filter_points(year_df, china_boundary)

    out_fp = os.path.join(OUT_DIR, f"china_dynamic_y{year}.parquet")
    year_df.to_parquet(out_fp, index=False)

    print(f"[DYNAMIC] Year {year}: rows={len(year_df):,}, grids={year_df['grid_id'].nunique():,}")
    print("          saved:", out_fp)

    all_dynamic_parts.append(year_df[["grid_id", "lon", "lat"]].drop_duplicates())


# ----------------------------
# 4. Extract static data
# ----------------------------

all_static_parts = []

for year in YEARS:
    files = list_files_for_year(DATA_DIR, year, "static")
    print(f"\n[STATIC] Year {year}: {len(files)} files")

    if len(files) == 0:
        continue

    year_parts = []

    for fp in files:
        try:
            df = read_csv_light(fp, STATIC_KEEP_COLS)

            if "lon" not in df.columns or "lat" not in df.columns:
                print("[SKIP] No lon/lat:", os.path.basename(fp))
                continue

            df = bbox_filter(df)

            if len(df) == 0:
                continue

            df = safe_keep_columns(df, STATIC_KEEP_COLS)
            year_parts.append(df)

        except Exception as e:
            print("[ERROR]", os.path.basename(fp), repr(e))

    if len(year_parts) == 0:
        print(f"[STATIC] Year {year}: no China bbox rows")
        continue

    year_df = pd.concat(year_parts, ignore_index=True)
    year_df = polygon_filter_points(year_df, china_boundary)

    # static table may contain duplicate grid rows; keep one per grid-year
    subset_keys = [c for c in ["grid_id", "year_static"] if c in year_df.columns]
    if subset_keys:
        year_df = year_df.drop_duplicates(subset=subset_keys)

    out_fp = os.path.join(OUT_DIR, f"china_static_y{year}.parquet")
    year_df.to_parquet(out_fp, index=False)

    print(f"[STATIC] Year {year}: rows={len(year_df):,}, grids={year_df['grid_id'].nunique():,}")
    print("         saved:", out_fp)

    all_static_parts.append(year_df[["grid_id", "lon", "lat"]].drop_duplicates())


# ----------------------------
# 5. Save grid inventory
# ----------------------------

grid_parts = []

if all_dynamic_parts:
    grid_parts.extend(all_dynamic_parts)

if all_static_parts:
    grid_parts.extend(all_static_parts)

if grid_parts:
    grid_inventory = pd.concat(grid_parts, ignore_index=True).drop_duplicates("grid_id")
    grid_inventory = grid_inventory.sort_values(["lat", "lon"]).reset_index(drop=True)

    grid_out = os.path.join(OUT_DIR, "china_grid_inventory.parquet")
    grid_inventory.to_parquet(grid_out, index=False)

    csv_out = os.path.join(OUT_DIR, "china_grid_inventory.csv")
    grid_inventory.to_csv(csv_out, index=False)

    print("\n[OK] China grid inventory")
    print("     grids:", len(grid_inventory))
    print("     parquet:", grid_out)
    print("     csv:", csv_out)
else:
    print("\n[WARN] No China grid inventory generated.")


# ----------------------------
# 6. Quick summary
# ----------------------------

print("\nDone.")
print("Output folder:", OUT_DIR)

In [ ]:
# ============================================================
# Step 1B. Check China extracted data completeness
# ============================================================

import os
import glob
import pandas as pd
import numpy as np

OUT_DIR = "/content/drive/MyDrive/GEE_China_VegClimate_NO"

dyn_files = sorted(glob.glob(os.path.join(OUT_DIR, "china_dynamic_y*.parquet")))
sta_files = sorted(glob.glob(os.path.join(OUT_DIR, "china_static_y*.parquet")))

print("Dynamic files:", len(dyn_files))
print("Static files:", len(sta_files))

# ---- dynamic summary by year ----
dyn_summary = []

for fp in dyn_files:
    df = pd.read_parquet(fp)

    year = int(df["year"].iloc[0]) if "year" in df.columns and len(df) > 0 else None

    row = {
        "year": year,
        "rows": len(df),
        "n_grid": df["grid_id"].nunique(),
        "n_step": df["step16"].nunique() if "step16" in df.columns else np.nan,
        "lon_min": df["lon"].min(),
        "lon_max": df["lon"].max(),
        "lat_min": df["lat"].min(),
        "lat_max": df["lat"].max(),
    }

    for col in ["EVI", "LAI", "VPD", "LST_Day", "SM_rootproxy", "Precip", "Tair"]:
        if col in df.columns:
            row[f"{col}_valid_frac"] = df[col].notna().mean()
        else:
            row[f"{col}_valid_frac"] = np.nan

    dyn_summary.append(row)

dyn_summary = pd.DataFrame(dyn_summary).sort_values("year")
display(dyn_summary)

summary_fp = os.path.join(OUT_DIR, "china_dynamic_completeness_summary.csv")
dyn_summary.to_csv(summary_fp, index=False)
print("Saved:", summary_fp)


# ---- static summary by year ----
sta_summary = []

for fp in sta_files:
    df = pd.read_parquet(fp)

    year = int(df["year_static"].iloc[0]) if "year_static" in df.columns and len(df) > 0 else None

    row = {
        "year_static": year,
        "rows": len(df),
        "n_grid": df["grid_id"].nunique(),
        "lon_min": df["lon"].min(),
        "lon_max": df["lon"].max(),
        "lat_min": df["lat"].min(),
        "lat_max": df["lat"].max(),
    }

    for col in ["elevation", "slope", "LC_Type1", "Greenup_1", "Peak_1", "Dormancy_1"]:
        if col in df.columns:
            row[f"{col}_valid_frac"] = df[col].notna().mean()
        else:
            row[f"{col}_valid_frac"] = np.nan

    sta_summary.append(row)

sta_summary = pd.DataFrame(sta_summary).sort_values("year_static")
display(sta_summary)

summary_fp = os.path.join(OUT_DIR, "china_static_completeness_summary.csv")
sta_summary.to_csv(summary_fp, index=False)
print("Saved:", summary_fp)

In [ ]:
# ============================================================
# Step 1C. Quick spatial visualization of extracted China grids
# ============================================================

import os
import pandas as pd
import matplotlib.pyplot as plt

OUT_DIR = "/content/drive/MyDrive/GEE_China_VegClimate_NO"
grid_fp = os.path.join(OUT_DIR, "china_grid_inventory.parquet")

grid = pd.read_parquet(grid_fp)

plt.figure(figsize=(8, 7))
plt.scatter(grid["lon"], grid["lat"], s=2, alpha=0.6)
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.title("Extracted China grid centers")
plt.xlim(70, 138)
plt.ylim(15, 56)
plt.grid(True, linewidth=0.3, alpha=0.4)
plt.show()

print("Number of grids:", len(grid))
print(grid[["lon", "lat"]].describe())

In [ ]:
# ============================================================
# Download China boundary and save to Google Drive
# Priority:
#   1) GADM China ADM0 GeoPackage
#   2) Natural Earth Admin0 fallback
#
# Output:
#   /content/drive/MyDrive/China_boundary/china_boundary.gpkg
# ============================================================

import os
import sys
import zipfile
import subprocess
import requests
import pandas as pd

# ----------------------------
# 0. Install/check packages
# ----------------------------
try:
    import geopandas as gpd
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "geopandas", "pyogrio"])
    import geopandas as gpd

# ----------------------------
# 1. Settings
# ----------------------------
OUT_DIR = "/content/drive/MyDrive/China_boundary"
os.makedirs(OUT_DIR, exist_ok=True)

OUT_GPKG = os.path.join(OUT_DIR, "china_boundary.gpkg")
TMP_DIR = os.path.join(OUT_DIR, "_tmp_download")
os.makedirs(TMP_DIR, exist_ok=True)

# GADM v4.1 direct GeoPackage URL
# If this URL changes later, the code will fall back to Natural Earth.
GADM_URL = "https://geodata.ucdavis.edu/gadm/gadm4.1/gpkg/gadm41_CHN.gpkg"

# Natural Earth 10m Admin 0 Countries fallback
NE_URL = "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip"

# ----------------------------
# 2. Helper functions
# ----------------------------
def download_file(url, out_path, timeout=120):
    print(f"Downloading:\n  {url}")
    r = requests.get(url, stream=True, timeout=timeout)
    r.raise_for_status()

    total = int(r.headers.get("content-length", 0))
    downloaded = 0

    with open(out_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)

    mb = downloaded / 1024 / 1024
    print(f"Saved: {out_path} ({mb:.2f} MB)")


def save_boundary(gdf, out_gpkg):
    gdf = gdf.to_crs("EPSG:4326")

    # Dissolve all features into one multipolygon
    gdf["dissolve_id"] = 1
    gdf = gdf.dissolve(by="dissolve_id").reset_index(drop=True)

    # Clean minimal attributes
    gdf["name"] = "China"
    gdf = gdf[["name", "geometry"]]

    # Save
    if os.path.exists(out_gpkg):
        os.remove(out_gpkg)

    gdf.to_file(out_gpkg, layer="china_adm0", driver="GPKG")
    print(f"\n[OK] China boundary saved to:\n  {out_gpkg}")
    print("CRS:", gdf.crs)
    print("Bounds:", gdf.total_bounds)
    return gdf


# ----------------------------
# 3. Try GADM first
# ----------------------------
success = False

try:
    gadm_path = os.path.join(TMP_DIR, "gadm41_CHN.gpkg")
    download_file(GADM_URL, gadm_path)

    # GADM gpkg usually contains multiple layers: ADM_0, ADM_1, etc.
    layers = gpd.list_layers(gadm_path)
    print("\nGADM layers:")
    print(layers)

    # Pick ADM0 layer. Layer name can vary slightly.
    layer_names = list(layers["name"])
    adm0_candidates = [x for x in layer_names if "ADM_0" in x or x.endswith("_0") or x.lower().endswith("0")]

    if len(adm0_candidates) == 0:
        layer = layer_names[0]
        print(f"[WARN] ADM0 layer not clearly found. Using first layer: {layer}")
    else:
        layer = adm0_candidates[0]
        print(f"Using GADM layer: {layer}")

    china = gpd.read_file(gadm_path, layer=layer)
    china = save_boundary(china, OUT_GPKG)
    success = True

except Exception as e:
    print("\n[WARN] GADM download/read failed. Falling back to Natural Earth.")
    print("Error:", repr(e))


# ----------------------------
# 4. Fallback: Natural Earth
# ----------------------------
if not success:
    try:
        ne_zip = os.path.join(TMP_DIR, "ne_10m_admin_0_countries.zip")
        download_file(NE_URL, ne_zip)

        extract_dir = os.path.join(TMP_DIR, "ne_10m_admin_0_countries")
        os.makedirs(extract_dir, exist_ok=True)

        with zipfile.ZipFile(ne_zip, "r") as z:
            z.extractall(extract_dir)

        shp_files = [os.path.join(extract_dir, f) for f in os.listdir(extract_dir) if f.endswith(".shp")]
        if len(shp_files) == 0:
            raise FileNotFoundError("No shapefile found in Natural Earth zip.")

        world = gpd.read_file(shp_files[0]).to_crs("EPSG:4326")
        print("\nNatural Earth columns:")
        print(world.columns.tolist())

        # Try robust China filtering.
        # Natural Earth usually has NAME, ADMIN, SOVEREIGNT, ADM0_A3, ISO_A3.
        mask = pd.Series(False, index=world.index)

        for col in ["ADMIN", "NAME", "NAME_LONG", "SOVEREIGNT", "BRK_NAME"]:
            if col in world.columns:
                mask = mask | world[col].astype(str).str.contains("China", case=False, na=False)

        for col in ["ADM0_A3", "ISO_A3", "GU_A3", "SOV_A3"]:
            if col in world.columns:
                mask = mask | world[col].astype(str).isin(["CHN"])

        china = world[mask].copy()

        if len(china) == 0:
            raise ValueError("Could not find China in Natural Earth attributes.")

        print("\nNatural Earth selected features:")
        show_cols = [c for c in ["ADMIN", "NAME", "SOVEREIGNT", "ADM0_A3", "ISO_A3"] if c in china.columns]
        print(china[show_cols])

        china = save_boundary(china, OUT_GPKG)
        success = True

    except Exception as e:
        print("\n[ERROR] Natural Earth fallback also failed.")
        print("Error:", repr(e))
        raise e


# ----------------------------
# 5. Quick plot
# ----------------------------
import matplotlib.pyplot as plt

china = gpd.read_file(OUT_GPKG, layer="china_adm0").to_crs("EPSG:4326")

fig, ax = plt.subplots(figsize=(7, 7))
china.boundary.plot(ax=ax, linewidth=0.8)
ax.set_xlim(70, 138)
ax.set_ylim(15, 56)
ax.set_title("Downloaded China boundary")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.grid(alpha=0.3)
plt.show()

print("\nFinal boundary file:")
print(OUT_GPKG)

In [ ]:
# ============================================================
# Download and merge China boundary with Taiwan included
# Source: GADM v4.1 GeoPackage
#
# Output:
#   /content/drive/MyDrive/China_boundary/china_boundary_with_taiwan.gpkg
# ============================================================

import os
import sys
import subprocess
import requests

# ----------------------------
# 0. Install/check packages
# ----------------------------
try:
    import geopandas as gpd
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "geopandas", "pyogrio"])
    import geopandas as gpd

# ----------------------------
# 1. Settings
# ----------------------------
OUT_DIR = "/content/drive/MyDrive/China_boundary"
TMP_DIR = os.path.join(OUT_DIR, "_tmp_download")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

OUT_GPKG = os.path.join(OUT_DIR, "china_boundary_with_taiwan.gpkg")

GADM_URLS = {
    "CHN": "https://geodata.ucdavis.edu/gadm/gadm4.1/gpkg/gadm41_CHN.gpkg",
    "TWN": "https://geodata.ucdavis.edu/gadm/gadm4.1/gpkg/gadm41_TWN.gpkg",
}

# ----------------------------
# 2. Helper functions
# ----------------------------
def download_file(url, out_path, timeout=180):
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        print(f"[EXISTS] {out_path}")
        return

    print(f"Downloading:\n  {url}")
    r = requests.get(url, stream=True, timeout=timeout)
    r.raise_for_status()

    with open(out_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)

    print(f"[OK] Saved: {out_path} ({os.path.getsize(out_path)/1024/1024:.2f} MB)")


def read_adm0_gadm(gpkg_path):
    layers = gpd.list_layers(gpkg_path)
    layer_names = list(layers["name"])

    # GADM layers are often like ADM_ADM_0, ADM_ADM_1, etc.
    adm0_candidates = [
        x for x in layer_names
        if "ADM_0" in x or x.endswith("_0") or x.lower().endswith("0")
    ]

    if len(adm0_candidates) == 0:
        layer = layer_names[0]
        print(f"[WARN] ADM0 layer not clearly found. Using first layer: {layer}")
    else:
        layer = adm0_candidates[0]

    print(f"Reading layer: {layer}")
    gdf = gpd.read_file(gpkg_path, layer=layer).to_crs("EPSG:4326")
    return gdf


# ----------------------------
# 3. Download and read CHN + TWN
# ----------------------------
parts = []

for code, url in GADM_URLS.items():
    local_path = os.path.join(TMP_DIR, f"gadm41_{code}.gpkg")
    download_file(url, local_path)
    gdf = read_adm0_gadm(local_path)
    gdf["source_code"] = code
    parts.append(gdf)

china_all = gpd.GeoDataFrame(
    gpd.pd.concat(parts, ignore_index=True),
    crs="EPSG:4326"
)

# ----------------------------
# 4. Dissolve into one study boundary
# ----------------------------
china_all["study_area"] = "China_with_Taiwan"
china_dissolved = china_all.dissolve(by="study_area").reset_index()
china_dissolved["name"] = "China"
china_dissolved = china_dissolved[["name", "geometry"]].to_crs("EPSG:4326")

# ----------------------------
# 5. Save
# ----------------------------
if os.path.exists(OUT_GPKG):
    os.remove(OUT_GPKG)

china_dissolved.to_file(OUT_GPKG, layer="china_adm0_with_taiwan", driver="GPKG")

print("\n[OK] Saved merged China boundary including Taiwan:")
print(OUT_GPKG)
print("Bounds:", china_dissolved.total_bounds)

# ----------------------------
# 6. Quick plot
# ----------------------------
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 7))
china_dissolved.boundary.plot(ax=ax, linewidth=0.8)
ax.set_xlim(70, 138)
ax.set_ylim(15, 56)
ax.set_title("China study boundary with Taiwan included")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# Download and merge full China study boundary:
#   CHN + TWN + HKG + MAC
#
# Output:
#   /content/drive/MyDrive/China_boundary/china_boundary_full.gpkg
#
# Note:
#   This is for defining the research study boundary consistently.
# ============================================================

import os
import sys
import subprocess
import requests

try:
    import geopandas as gpd
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "geopandas", "pyogrio"])
    import geopandas as gpd

import pandas as pd
import matplotlib.pyplot as plt

# ----------------------------
# 1. Paths
# ----------------------------
OUT_DIR = "/content/drive/MyDrive/China_boundary"
TMP_DIR = os.path.join(OUT_DIR, "_tmp_download")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

OUT_GPKG = os.path.join(OUT_DIR, "china_boundary_full.gpkg")

# GADM v4.1 GeoPackage URLs
GADM_URLS = {
    "CHN": "https://geodata.ucdavis.edu/gadm/gadm4.1/gpkg/gadm41_CHN.gpkg",
    "TWN": "https://geodata.ucdavis.edu/gadm/gadm4.1/gpkg/gadm41_TWN.gpkg",
    "HKG": "https://geodata.ucdavis.edu/gadm/gadm4.1/gpkg/gadm41_HKG.gpkg",
    "MAC": "https://geodata.ucdavis.edu/gadm/gadm4.1/gpkg/gadm41_MAC.gpkg",
}

# ----------------------------
# 2. Helper functions
# ----------------------------
def download_file(url, out_path, timeout=180):
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        print(f"[EXISTS] {out_path}")
        return True

    print(f"Downloading:\n  {url}")
    try:
        r = requests.get(url, stream=True, timeout=timeout)
        r.raise_for_status()
        with open(out_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
        print(f"[OK] Saved: {out_path} ({os.path.getsize(out_path)/1024/1024:.2f} MB)")
        return True
    except Exception as e:
        print(f"[FAILED] {url}")
        print("Error:", repr(e))
        return False


def read_adm0_gadm(gpkg_path):
    layers = gpd.list_layers(gpkg_path)
    layer_names = list(layers["name"])

    # GADM layers are often like ADM_ADM_0 / ADM_0 / gadm41_CHN_0
    adm0_candidates = [
        x for x in layer_names
        if "ADM_0" in x or x.endswith("_0") or x.lower().endswith("0")
    ]

    if len(adm0_candidates) == 0:
        layer = layer_names[0]
        print(f"[WARN] ADM0 layer not clearly found. Using first layer: {layer}")
    else:
        layer = adm0_candidates[0]

    print(f"Reading layer: {layer}")
    gdf = gpd.read_file(gpkg_path, layer=layer).to_crs("EPSG:4326")
    return gdf


# ----------------------------
# 3. Download and read all parts
# ----------------------------
parts = []

for code, url in GADM_URLS.items():
    local_path = os.path.join(TMP_DIR, f"gadm41_{code}.gpkg")

    ok = download_file(url, local_path)
    if not ok:
        print(f"[WARN] Skipping {code}; download failed.")
        continue

    try:
        gdf = read_adm0_gadm(local_path)
        gdf["source_code"] = code
        parts.append(gdf)
        print(f"[OK] Loaded {code}: {len(gdf)} feature(s), bounds={gdf.total_bounds}")
    except Exception as e:
        print(f"[WARN] Failed to read {code}")
        print("Error:", repr(e))

if len(parts) == 0:
    raise RuntimeError("No boundary data loaded. Please check network or GADM URLs.")

all_gdf = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs="EPSG:4326")

print("\nLoaded source parts:")
print(all_gdf[["source_code"]].value_counts())

# ----------------------------
# 4. Save both individual parts and dissolved full boundary
# ----------------------------
if os.path.exists(OUT_GPKG):
    os.remove(OUT_GPKG)

# Save source parts
parts_out = all_gdf[["source_code", "geometry"]].copy()
parts_out.to_file(OUT_GPKG, layer="china_parts_CHN_TWN_HKG_MAC", driver="GPKG")

# Dissolve into one study boundary
all_gdf["study_area"] = "China_full_study_boundary"
dissolved = all_gdf.dissolve(by="study_area").reset_index()
dissolved["name"] = "China"
dissolved = dissolved[["name", "geometry"]].to_crs("EPSG:4326")
dissolved.to_file(OUT_GPKG, layer="china_full", driver="GPKG")

print("\n[OK] Saved full China study boundary:")
print(OUT_GPKG)
print("Layers: china_parts_CHN_TWN_HKG_MAC, china_full")
print("Full bounds:", dissolved.total_bounds)

# ----------------------------
# 5. Quick plots
# ----------------------------
fig, ax = plt.subplots(figsize=(8, 7))
dissolved.boundary.plot(ax=ax, linewidth=0.8)
ax.set_xlim(70, 138)
ax.set_ylim(15, 56)
ax.set_title("Full China study boundary: CHN + TWN + HKG + MAC")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.3)
plt.show()

# Zoom: Hong Kong / Macau / Pearl River Delta
fig, ax = plt.subplots(figsize=(6, 5))
parts_out.boundary.plot(ax=ax, linewidth=0.8)
ax.set_xlim(112.5, 115.5)
ax.set_ylim(21.5, 23.5)
ax.set_title("Hong Kong and Macau check")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.3)
plt.show()

# Zoom: Taiwan
fig, ax = plt.subplots(figsize=(5, 6))
parts_out.boundary.plot(ax=ax, linewidth=0.8)
ax.set_xlim(118, 123)
ax.set_ylim(21, 26.5)
ax.set_title("Taiwan check")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# Create boundary_full parquet files from existing bbox parquet files
# using full China boundary: CHN + TWN + HKG + MAC
# ============================================================

import os
import glob
import pandas as pd
import geopandas as gpd

IN_DIR = "/content/drive/MyDrive/GEE_China_VegClimate_NO"
OUT_DIR = "/content/drive/MyDrive/GEE_China_VegClimate_NO_boundary_full"
CHINA_BOUNDARY_PATH = "/content/drive/MyDrive/China_boundary/china_boundary_full.gpkg"

os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------
# 1. Check input files
# ----------------------------
bbox_dyn = sorted(glob.glob(os.path.join(IN_DIR, "china_dynamic_y*.parquet")))
bbox_sta = sorted(glob.glob(os.path.join(IN_DIR, "china_static_y*.parquet")))

print("Input dynamic files:", len(bbox_dyn))
print("Input static files:", len(bbox_sta))

if len(bbox_dyn) == 0 or len(bbox_sta) == 0:
    raise FileNotFoundError(
        f"No bbox parquet files found in {IN_DIR}. "
        "Please check the previous extraction output folder."
    )

if not os.path.exists(CHINA_BOUNDARY_PATH):
    raise FileNotFoundError(
        f"Boundary file not found: {CHINA_BOUNDARY_PATH}. "
        "Please run the CHN+TWN+HKG+MAC boundary download/merge code first."
    )

# ----------------------------
# 2. Load China full boundary
# ----------------------------
try:
    china = gpd.read_file(CHINA_BOUNDARY_PATH, layer="china_full").to_crs("EPSG:4326")
except Exception:
    china = gpd.read_file(CHINA_BOUNDARY_PATH).to_crs("EPSG:4326")

china = china.dissolve().reset_index(drop=True)

# Optional: small buffer for tiny regions like Macau/Hong Kong at 0.1 degree grid
# 0.03 degree is roughly 3 km; adjust only if needed.
USE_SMALL_BUFFER = True
if USE_SMALL_BUFFER:
    china_filter = china.copy()
    china_filter["geometry"] = china_filter.geometry.buffer(0.03)
else:
    china_filter = china

# ----------------------------
# 3. Filter function
# ----------------------------
def filter_file(fp, out_fp):
    df = pd.read_parquet(fp)

    if len(df) == 0:
        print("[SKIP empty]", os.path.basename(fp))
        return None

    if "lon" not in df.columns or "lat" not in df.columns:
        print("[SKIP no lon/lat]", os.path.basename(fp))
        return None

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs="EPSG:4326"
    )

    # Use within buffered boundary for grid centers
    gdf2 = gpd.sjoin(
        gdf,
        china_filter[["geometry"]],
        how="inner",
        predicate="within"
    )

    drop_cols = [c for c in ["geometry", "index_right"] if c in gdf2.columns]
    out = pd.DataFrame(gdf2.drop(columns=drop_cols))

    out.to_parquet(out_fp, index=False)

    info = {
        "file": os.path.basename(fp),
        "out_file": os.path.basename(out_fp),
        "rows_before": len(df),
        "rows_after": len(out),
        "grids_before": df["grid_id"].nunique() if "grid_id" in df.columns else None,
        "grids_after": out["grid_id"].nunique() if "grid_id" in out.columns else None,
    }

    print(
        f"[OK] {info['file']}: "
        f"rows {info['rows_before']:,} -> {info['rows_after']:,}; "
        f"grids {info['grids_before']:,} -> {info['grids_after']:,}"
    )

    return info

# ----------------------------
# 4. Process files
# ----------------------------
summary = []

for fp in bbox_dyn:
    out_name = os.path.basename(fp).replace(".parquet", "_boundary_full.parquet")
    out_fp = os.path.join(OUT_DIR, out_name)
    info = filter_file(fp, out_fp)
    if info:
        info["kind"] = "dynamic"
        summary.append(info)

for fp in bbox_sta:
    out_name = os.path.basename(fp).replace(".parquet", "_boundary_full.parquet")
    out_fp = os.path.join(OUT_DIR, out_name)
    info = filter_file(fp, out_fp)
    if info:
        info["kind"] = "static"
        summary.append(info)

summary_df = pd.DataFrame(summary)
summary_fp = os.path.join(OUT_DIR, "boundary_full_filter_summary.csv")
summary_df.to_csv(summary_fp, index=False)

print("\n[OK] Saved summary:", summary_fp)
display(summary_df)

In [ ]:
# ============================================================
# Build grid inventory for boundary_full dataset
# ============================================================

import os
import glob
import pandas as pd

OUT_DIR = "/content/drive/MyDrive/GEE_China_VegClimate_NO_boundary_full"

parts = []

for fp in glob.glob(os.path.join(OUT_DIR, "*boundary_full.parquet")):
    df = pd.read_parquet(fp, columns=["grid_id", "lon", "lat"])
    parts.append(df.drop_duplicates())

if len(parts) == 0:
    raise FileNotFoundError(f"No boundary_full parquet files found in {OUT_DIR}")

grid = pd.concat(parts, ignore_index=True).drop_duplicates("grid_id")
grid = grid.sort_values(["lat", "lon"]).reset_index(drop=True)

grid.to_parquet(os.path.join(OUT_DIR, "china_grid_inventory_full.parquet"), index=False)
grid.to_csv(os.path.join(OUT_DIR, "china_grid_inventory_full.csv"), index=False)

print("Final China full grids:", len(grid))
print("Saved to:", OUT_DIR)
display(grid.head())

In [ ]:
# ============================================================
# Step 2. Build China 16-day core panel
# Merge dynamic + static data after full China boundary filtering
#
# Input:
#   china_dynamic_y*_china_full.parquet or similar
#   china_static_y*_china_full.parquet or similar
#
# Output:
#   china_panel_16day_core.parquet
# ============================================================

import os
import glob
import re
import numpy as np
import pandas as pd

# ----------------------------
# 0. Paths
# ----------------------------
IN_DIR = "/content/drive/MyDrive/GEE_China_VegClimate_NO_boundary_full"
OUT_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset"
os.makedirs(OUT_DIR, exist_ok=True)

# ----------------------------
# 1. Find files robustly
# ----------------------------
dyn_files = sorted(glob.glob(os.path.join(IN_DIR, "*dynamic*.parquet")))
sta_files = sorted(glob.glob(os.path.join(IN_DIR, "*static*.parquet")))

print("Dynamic files:", len(dyn_files))
print("Static files:", len(sta_files))

if len(dyn_files) == 0:
    raise FileNotFoundError(f"No dynamic parquet files found in {IN_DIR}")

if len(sta_files) == 0:
    raise FileNotFoundError(f"No static parquet files found in {IN_DIR}")


def extract_year_from_name(fp):
    name = os.path.basename(fp)
    m = re.search(r"y(20\d{2})", name)
    if m:
        return int(m.group(1))
    return None


# ----------------------------
# 2. Core variables
# ----------------------------
dyn_cols = [
    "grid_id", "lon", "lat", "region_id",
    "year", "step16", "date_start", "date_end", "doy_start", "doy_end",
    "EVI", "LAI",
    "VPD", "LST_Day", "SM_rootproxy", "Precip", "Tair"
]

sta_cols = [
    "grid_id", "lon", "lat", "region_id",
    "year_static",
    "elevation", "slope", "LC_Type1",
    "lc_veg_frac", "evi_peak_frac", "evi_amp_frac", "valid_frac",
    "Greenup_1", "Peak_1", "Dormancy_1"
]


def read_keep(fp, cols):
    df = pd.read_parquet(fp)
    keep = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f"[WARN] {os.path.basename(fp)} missing columns: {missing}")
    return df[keep].copy()


# ----------------------------
# 3. Load dynamic
# ----------------------------
dyn_parts = []

for fp in dyn_files:
    year = extract_year_from_name(fp)
    df = read_keep(fp, dyn_cols)

    if "year" not in df.columns and year is not None:
        df["year"] = year

    # remove duplicated rows if any
    subset = [c for c in ["grid_id", "year", "step16"] if c in df.columns]
    if subset:
        df = df.drop_duplicates(subset=subset)

    dyn_parts.append(df)
    print(f"[DYN] {os.path.basename(fp)} rows={len(df):,}, grids={df['grid_id'].nunique():,}")

dynamic = pd.concat(dyn_parts, ignore_index=True)

print("\nDynamic total:")
print(dynamic.shape)
print("Years:", sorted(dynamic["year"].dropna().unique()))
print("Grids:", dynamic["grid_id"].nunique())


# ----------------------------
# 4. Load static
# ----------------------------
sta_parts = []

for fp in sta_files:
    year = extract_year_from_name(fp)
    df = read_keep(fp, sta_cols)

    if "year_static" not in df.columns and year is not None:
        df["year_static"] = year

    subset = [c for c in ["grid_id", "year_static"] if c in df.columns]
    if subset:
        df = df.drop_duplicates(subset=subset)

    sta_parts.append(df)
    print(f"[STA] {os.path.basename(fp)} rows={len(df):,}, grids={df['grid_id'].nunique():,}")

static = pd.concat(sta_parts, ignore_index=True)

print("\nStatic total:")
print(static.shape)
print("Years:", sorted(static["year_static"].dropna().unique()))
print("Grids:", static["grid_id"].nunique())


# ----------------------------
# 5. Merge dynamic + static by grid_id and year
# ----------------------------
static_ren = static.rename(columns={"year_static": "year"})

# Avoid duplicated lon/lat/region columns from static
drop_from_static = [c for c in ["lon", "lat", "region_id"] if c in static_ren.columns]
static_ren = static_ren.drop(columns=drop_from_static)

panel = dynamic.merge(
    static_ren,
    on=["grid_id", "year"],
    how="left",
    validate="many_to_one"
)

print("\nMerged panel:")
print(panel.shape)
print("Years:", sorted(panel["year"].dropna().unique()))
print("Grids:", panel["grid_id"].nunique())

# ----------------------------
# 6. Basic cleaning
# ----------------------------
# ensure numeric
num_cols = [
    "lon", "lat", "year", "step16", "doy_start", "doy_end",
    "EVI", "LAI", "VPD", "LST_Day", "SM_rootproxy", "Precip", "Tair",
    "elevation", "slope", "LC_Type1"
]
for c in num_cols:
    if c in panel.columns:
        panel[c] = pd.to_numeric(panel[c], errors="coerce")

# season labels using middle DOY
panel["doy_mid"] = (panel["doy_start"] + panel["doy_end"]) / 2.0

def season_from_doy(doy):
    if pd.isna(doy):
        return np.nan
    if 60 <= doy <= 151:
        return "spring"       # Mar-May
    elif 152 <= doy <= 243:
        return "summer"       # Jun-Aug
    elif 213 <= doy <= 273:
        return "late_summer"  # Aug-Sep, overlapping definition handled later
    elif 244 <= doy <= 304:
        return "autumn"       # Sep-Oct
    else:
        return "other"

panel["season_simple"] = panel["doy_mid"].apply(season_from_doy)

# save
out_fp = os.path.join(OUT_DIR, "china_panel_16day_core.parquet")
panel.to_parquet(out_fp, index=False)

print("\n[OK] Saved core 16-day panel:")
print(out_fp)
print(panel.head())

In [ ]:
# ============================================================
# Check core panel completeness before anomaly construction
# ============================================================

import os
import pandas as pd
import numpy as np

OUT_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset"
panel_fp = os.path.join(OUT_DIR, "china_panel_16day_core.parquet")

panel = pd.read_parquet(panel_fp)

core_vars = [
    "EVI", "LAI",
    "VPD", "LST_Day", "SM_rootproxy", "Precip", "Tair",
    "elevation", "slope", "LC_Type1"
]

summary = []

for year, g in panel.groupby("year"):
    row = {
        "year": int(year),
        "rows": len(g),
        "grids": g["grid_id"].nunique(),
        "steps": g["step16"].nunique()
    }
    for v in core_vars:
        if v in g.columns:
            row[f"{v}_valid"] = g[v].notna().mean()
        else:
            row[f"{v}_valid"] = np.nan
    summary.append(row)

summary = pd.DataFrame(summary)
display(summary)

summary.to_csv(os.path.join(OUT_DIR, "core_panel_completeness_by_year.csv"), index=False)

# Overall variable completeness
overall = []
for v in core_vars:
    overall.append({
        "var": v,
        "valid_frac": panel[v].notna().mean() if v in panel.columns else np.nan,
        "mean": panel[v].mean() if v in panel.columns else np.nan,
        "std": panel[v].std() if v in panel.columns else np.nan,
        "min": panel[v].min() if v in panel.columns else np.nan,
        "max": panel[v].max() if v in panel.columns else np.nan,
    })

overall = pd.DataFrame(overall)
display(overall)
overall.to_csv(os.path.join(OUT_DIR, "core_panel_variable_summary.csv"), index=False)

In [ ]:
# ============================================================
# Step 3. Compute 16-day grid-specific anomalies and z-scores
# Clean and robust version
# ============================================================

import os
import numpy as np
import pandas as pd

OUT_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset"
in_fp = os.path.join(OUT_DIR, "china_panel_16day_core.parquet")

panel = pd.read_parquet(in_fp)

anom_vars = [
    "EVI", "LAI",
    "VPD", "LST_Day", "SM_rootproxy", "Precip", "Tair"
]

for c in anom_vars:
    panel[c] = pd.to_numeric(panel[c], errors="coerce")

panel.loc[(panel["EVI"] < -0.2) | (panel["EVI"] > 1.2), "EVI"] = np.nan
panel.loc[(panel["LAI"] < 0) | (panel["LAI"] > 12), "LAI"] = np.nan

# One-shot climatology table
clim = panel.groupby(["grid_id", "step16"])[anom_vars].agg(["mean", "std", "count"])
clim.columns = [f"{var}_clim_{stat}" for var, stat in clim.columns]
clim = clim.reset_index()

# Merge once
panel2 = panel.merge(
    clim,
    on=["grid_id", "step16"],
    how="left",
    validate="many_to_one"
)

for v in anom_vars:
    mean_col = f"{v}_clim_mean"
    std_col = f"{v}_clim_std"

    panel2[f"{v}_anom"] = panel2[v] - panel2[mean_col]

    std = panel2[std_col].replace(0, np.nan)
    std = std.where(std > 1e-6, np.nan)

    panel2[f"{v}_z"] = panel2[f"{v}_anom"] / std
    panel2[f"{v}_z"] = panel2[f"{v}_z"].clip(-5, 5)

clim_fp = os.path.join(OUT_DIR, "china_16day_grid_climatology.parquet")
anom_fp = os.path.join(OUT_DIR, "china_panel_16day_anomaly.parquet")

clim.to_parquet(clim_fp, index=False)
panel2.to_parquet(anom_fp, index=False)

print("[OK] Saved climatology:", clim_fp)
print("[OK] Saved anomaly panel:", anom_fp)
print("Panel shape:", panel2.shape)

summary = []
for v in anom_vars:
    summary.append({
        "var": v,
        "raw_valid": panel2[v].notna().mean(),
        "anom_valid": panel2[f"{v}_anom"].notna().mean(),
        "z_valid": panel2[f"{v}_z"].notna().mean(),
        "mean_clim_count": panel2[f"{v}_clim_count"].mean(),
        "median_clim_count": panel2[f"{v}_clim_count"].median(),
        "z_mean": panel2[f"{v}_z"].mean(),
        "z_std": panel2[f"{v}_z"].std()
    })

summary = pd.DataFrame(summary)
display(summary)
summary.to_csv(os.path.join(OUT_DIR, "anomaly_variable_summary.csv"), index=False)

In [ ]:
# ============================================================
# Step 4. Build seasonal response panel
# X: spring vegetation state + summer heat-dryness forcing + static context
# Y: late-summer EVI/LAI anomaly response
# ============================================================

import os
import numpy as np
import pandas as pd

OUT_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset"
in_fp = os.path.join(OUT_DIR, "china_panel_16day_anomaly.parquet")

df = pd.read_parquet(in_fp)

# Ensure DOY middle
df["doy_mid"] = (pd.to_numeric(df["doy_start"], errors="coerce") +
                 pd.to_numeric(df["doy_end"], errors="coerce")) / 2.0

# Seasonal windows
season_defs = {
    "spring": (60, 151),        # Mar-May
    "summer": (152, 243),       # Jun-Aug
    "late_summer": (213, 273),  # Aug-Sep
    "autumn": (244, 304),       # Sep-Oct, optional
}

z_vars = [
    "EVI_z", "LAI_z",
    "VPD_z", "LST_Day_z", "SM_rootproxy_z", "Precip_z", "Tair_z"
]

anom_vars = [
    "EVI_anom", "LAI_anom",
    "VPD_anom", "LST_Day_anom", "SM_rootproxy_anom", "Precip_anom", "Tair_anom"
]

raw_vars = [
    "EVI", "LAI",
    "VPD", "LST_Day", "SM_rootproxy", "Precip", "Tair"
]

static_vars = [
    "elevation", "slope", "LC_Type1",
    "lc_veg_frac", "evi_peak_frac", "evi_amp_frac", "valid_frac",
    "Greenup_1", "Peak_1", "Dormancy_1"
]

season_parts = []

for season, (d0, d1) in season_defs.items():
    sub = df[(df["doy_mid"] >= d0) & (df["doy_mid"] <= d1)].copy()

    agg_vars = [v for v in z_vars + anom_vars + raw_vars if v in sub.columns]

    agg = (
        sub
        .groupby(["grid_id", "year"], as_index=False)[agg_vars]
        .mean()
    )

    count = (
        sub
        .groupby(["grid_id", "year"], as_index=False)
        .size()
        .rename(columns={"size": f"{season}_n_steps"})
    )

    agg = agg.merge(count, on=["grid_id", "year"], how="left")

    rename = {v: f"{season}_{v}" for v in agg_vars}
    agg = agg.rename(columns=rename)

    season_parts.append(agg)

seasonal = season_parts[0]
for part in season_parts[1:]:
    seasonal = seasonal.merge(part, on=["grid_id", "year"], how="outer")

# Add static info
static_cols = ["grid_id", "year", "lon", "lat", "region_id"] + [v for v in static_vars if v in df.columns]
static = df[static_cols].drop_duplicates(subset=["grid_id", "year"])
seasonal = seasonal.merge(static, on=["grid_id", "year"], how="left")

# Model columns
x_cols = [
    "spring_EVI_z",
    "spring_LAI_z",
    "summer_VPD_z",
    "summer_LST_Day_z",
    "summer_SM_rootproxy_z",
    "summer_Precip_z",
    "summer_Tair_z",
    "elevation",
    "slope",
    "LC_Type1"
]

y_cols = [
    "late_summer_EVI_z",
    "late_summer_LAI_z"
]

# Also keep raw-anomaly targets for interpretation
y_anom_cols = [
    "late_summer_EVI_anom",
    "late_summer_LAI_anom"
]

# Validity flags
required_x = [c for c in x_cols if c in seasonal.columns]
required_y_evi = ["late_summer_EVI_z"]
required_y_lai = ["late_summer_LAI_z"]

seasonal["valid_x"] = seasonal[required_x].notna().all(axis=1)
seasonal["valid_y_evi"] = seasonal[required_y_evi].notna().all(axis=1)
seasonal["valid_y_lai"] = seasonal[required_y_lai].notna().all(axis=1)
seasonal["valid_model_any_y"] = seasonal["valid_x"] & (seasonal["valid_y_evi"] | seasonal["valid_y_lai"])
seasonal["valid_model_both_y"] = seasonal["valid_x"] & seasonal["valid_y_evi"] & seasonal["valid_y_lai"]

out_fp = os.path.join(OUT_DIR, "china_seasonal_response_panel.parquet")
seasonal.to_parquet(out_fp, index=False)

print("[OK] Saved seasonal response panel:", out_fp)
print("Rows:", len(seasonal))
print("Years:", sorted(seasonal["year"].dropna().unique()))
print("Grids:", seasonal["grid_id"].nunique())
print("Valid X:", seasonal["valid_x"].sum())
print("Valid X + any Y:", seasonal["valid_model_any_y"].sum())
print("Valid X + both Y:", seasonal["valid_model_both_y"].sum())

# Completeness
comp = []
for c in x_cols + y_cols + y_anom_cols:
    if c in seasonal.columns:
        comp.append({"column": c, "valid_frac": seasonal[c].notna().mean()})
comp = pd.DataFrame(comp)
display(comp)
comp.to_csv(os.path.join(OUT_DIR, "seasonal_response_completeness.csv"), index=False)

display(seasonal.head())

In [ ]:
# ============================================================
# Step 5. Build regular 2D grid metadata for China response fields
# ============================================================

import os
import numpy as np
import pandas as pd

OUT_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset"
in_fp = os.path.join(OUT_DIR, "china_seasonal_response_panel.parquet")

df = pd.read_parquet(in_fp)

# Use all known grids from the seasonal panel
grid = (
    df[["grid_id", "lon", "lat"]]
    .dropna()
    .drop_duplicates("grid_id")
    .copy()
)

# Round to avoid floating point noise
grid["lon_r"] = grid["lon"].round(4)
grid["lat_r"] = grid["lat"].round(4)

lons = np.sort(grid["lon_r"].unique())
lats = np.sort(grid["lat_r"].unique())

# row: north to south; col: west to east
lon_to_col = {lon: i for i, lon in enumerate(lons)}
lat_to_row = {lat: i for i, lat in enumerate(lats[::-1])}

grid["col"] = grid["lon_r"].map(lon_to_col).astype(int)
grid["row"] = grid["lat_r"].map(lat_to_row).astype(int)

H = len(lats)
W = len(lons)

print("Grid tensor shape H x W:", H, W)
print("Active grids:", len(grid))
print("Bounding rectangle cells:", H * W)
print("Active fraction:", len(grid) / (H * W))

out_fp = os.path.join(OUT_DIR, "china_grid_tensor_metadata.parquet")
grid.to_parquet(out_fp, index=False)
grid.to_csv(os.path.join(OUT_DIR, "china_grid_tensor_metadata.csv"), index=False)

print("[OK] Saved:", out_fp)
display(grid.head())

In [ ]:
# ============================================================
# Step 6. Convert seasonal response panel to year-level gridded tensors
# Output:
#   X_yearly.npy: [N_year, Cx, H, W]
#   Y_yearly.npy: [N_year, Cy, H, W]
#   M_yearly.npy: [N_year, 1, H, W]
#   sample_index_yearly.csv
#   channel_info_yearly.json
# ============================================================

import os
import json
import numpy as np
import pandas as pd

OUT_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset"

panel_fp = os.path.join(OUT_DIR, "china_seasonal_response_panel.parquet")
grid_fp = os.path.join(OUT_DIR, "china_grid_tensor_metadata.parquet")

panel = pd.read_parquet(panel_fp)
grid = pd.read_parquet(grid_fp)

x_cols = [
    "spring_EVI_z",
    "spring_LAI_z",
    "summer_VPD_z",
    "summer_LST_Day_z",
    "summer_SM_rootproxy_z",
    "summer_Precip_z",
    "summer_Tair_z",
    "elevation",
    "slope",
    "LC_Type1"
]

y_cols = [
    "late_summer_EVI_z",
    "late_summer_LAI_z"
]

x_cols = [c for c in x_cols if c in panel.columns]
y_cols = [c for c in y_cols if c in panel.columns]

print("X channels:", x_cols)
print("Y channels:", y_cols)

# Normalize static variables only. Dynamic variables are already z-scores.
panel = panel.copy()

static_norm_stats = {}
for c in ["elevation", "slope", "LC_Type1"]:
    if c in panel.columns:
        mu = panel[c].mean()
        sd = panel[c].std()
        if sd == 0 or np.isnan(sd):
            sd = 1.0
        panel[c] = (panel[c] - mu) / sd
        static_norm_stats[c] = {"mean": float(mu), "std": float(sd)}

# Merge row/col
panel = panel.merge(grid[["grid_id", "row", "col"]], on="grid_id", how="left")

H = int(grid["row"].max()) + 1
W = int(grid["col"].max()) + 1

sample_years = sorted(panel["year"].dropna().unique())
N = len(sample_years)
Cx = len(x_cols)
Cy = len(y_cols)

X = np.full((N, Cx, H, W), np.nan, dtype=np.float32)
Y = np.full((N, Cy, H, W), np.nan, dtype=np.float32)
M = np.zeros((N, 1, H, W), dtype=np.float32)

records = []

for i, year in enumerate(sample_years):
    sub = panel[panel["year"] == year].copy()
    sub = sub.dropna(subset=["row", "col"])
    sub["row"] = sub["row"].astype(int)
    sub["col"] = sub["col"].astype(int)

    for _, r in sub.iterrows():
        rr = int(r["row"])
        cc = int(r["col"])

        x_valid = all(pd.notna(r[c]) for c in x_cols)
        y_valid = all(pd.notna(r[c]) for c in y_cols)

        for k, c in enumerate(x_cols):
            if pd.notna(r[c]):
                X[i, k, rr, cc] = np.float32(r[c])

        for k, c in enumerate(y_cols):
            if pd.notna(r[c]):
                Y[i, k, rr, cc] = np.float32(r[c])

        if x_valid and y_valid:
            M[i, 0, rr, cc] = 1.0

    records.append({
        "sample_id": i,
        "year": int(year),
        "valid_pixels": int(M[i, 0].sum())
    })

# Fill invalid values with 0; mask controls valid loss
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
Y = np.nan_to_num(Y, nan=0.0, posinf=0.0, neginf=0.0)

sample_index = pd.DataFrame(records)

np.save(os.path.join(OUT_DIR, "X_yearly.npy"), X)
np.save(os.path.join(OUT_DIR, "Y_yearly.npy"), Y)
np.save(os.path.join(OUT_DIR, "M_yearly.npy"), M)

sample_index.to_csv(os.path.join(OUT_DIR, "sample_index_yearly.csv"), index=False)

channel_info = {
    "x_cols": x_cols,
    "y_cols": y_cols,
    "static_norm_stats": static_norm_stats,
    "shape": {
        "N": N,
        "Cx": Cx,
        "Cy": Cy,
        "H": H,
        "W": W
    }
}

with open(os.path.join(OUT_DIR, "channel_info_yearly.json"), "w") as f:
    json.dump(channel_info, f, indent=2)

print("[OK] Saved yearly tensors to:", OUT_DIR)
print("X:", X.shape)
print("Y:", Y.shape)
print("M:", M.shape)
display(sample_index)

In [ ]:
# ============================================================
# Step 7. Build patch dataset for U-Net / FNO training
# Output:
#   X_patch.npy: [N_patch, Cx, P, P]
#   Y_patch.npy: [N_patch, Cy, P, P]
#   M_patch.npy: [N_patch, 1, P, P]
#   patch_index.csv
# ============================================================

import os
import numpy as np
import pandas as pd

OUT_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset"

X = np.load(os.path.join(OUT_DIR, "X_yearly.npy"))
Y = np.load(os.path.join(OUT_DIR, "Y_yearly.npy"))
M = np.load(os.path.join(OUT_DIR, "M_yearly.npy"))
sample_index = pd.read_csv(os.path.join(OUT_DIR, "sample_index_yearly.csv"))

PATCH_SIZE = 64
STRIDE = 16
MIN_VALID_FRAC = 0.10  # at least 10% valid pixels in a patch

N, Cx, H, W = X.shape
Cy = Y.shape[1]

X_patches = []
Y_patches = []
M_patches = []
records = []

patch_id = 0

for n in range(N):
    year = int(sample_index.loc[n, "year"])

    for r0 in range(0, H - PATCH_SIZE + 1, STRIDE):
        for c0 in range(0, W - PATCH_SIZE + 1, STRIDE):
            r1 = r0 + PATCH_SIZE
            c1 = c0 + PATCH_SIZE

            m_patch = M[n:n+1, :, r0:r1, c0:c1]  # [1,1,P,P]
            valid_frac = float(m_patch.mean())

            if valid_frac < MIN_VALID_FRAC:
                continue

            X_patches.append(X[n, :, r0:r1, c0:c1])
            Y_patches.append(Y[n, :, r0:r1, c0:c1])
            M_patches.append(M[n, :, r0:r1, c0:c1])

            records.append({
                "patch_id": patch_id,
                "year": year,
                "sample_id": n,
                "row0": r0,
                "col0": c0,
                "valid_frac": valid_frac,
                "valid_pixels": int(m_patch.sum())
            })
            patch_id += 1

X_patch = np.stack(X_patches).astype(np.float32)
Y_patch = np.stack(Y_patches).astype(np.float32)
M_patch = np.stack(M_patches).astype(np.float32)
patch_index = pd.DataFrame(records)

np.save(os.path.join(OUT_DIR, "X_patch.npy"), X_patch)
np.save(os.path.join(OUT_DIR, "Y_patch.npy"), Y_patch)
np.save(os.path.join(OUT_DIR, "M_patch.npy"), M_patch)
patch_index.to_csv(os.path.join(OUT_DIR, "patch_index.csv"), index=False)

print("[OK] Saved patch dataset")
print("X_patch:", X_patch.shape)
print("Y_patch:", Y_patch.shape)
print("M_patch:", M_patch.shape)
print("Patch count:", len(patch_index))
display(patch_index.head())
display(patch_index.groupby("year")["patch_id"].count().reset_index(name="n_patch"))

In [ ]:
# ============================================================
# Quick visualization of yearly tensor fields
# ============================================================

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset"

X = np.load(os.path.join(OUT_DIR, "X_yearly.npy"))
Y = np.load(os.path.join(OUT_DIR, "Y_yearly.npy"))
M = np.load(os.path.join(OUT_DIR, "M_yearly.npy"))
sample_index = pd.read_csv(os.path.join(OUT_DIR, "sample_index_yearly.csv"))

with open(os.path.join(OUT_DIR, "channel_info_yearly.json"), "r") as f:
    info = json.load(f)

x_cols = info["x_cols"]
y_cols = info["y_cols"]

print("X channels:", x_cols)
print("Y channels:", y_cols)

def show_field(arr, mask, title, vmin=-2, vmax=2):
    a = arr.copy()
    a[mask <= 0] = np.nan
    plt.figure(figsize=(7, 5))
    plt.imshow(a, vmin=vmin, vmax=vmax)
    plt.colorbar(label="z-score")
    plt.title(title)
    plt.axis("off")
    plt.show()

# Choose one year, e.g. 2022
target_year = 2022
idx = sample_index.index[sample_index["year"] == target_year][0]

mask = M[idx, 0]

for ch in ["summer_VPD_z", "summer_LST_Day_z", "summer_SM_rootproxy_z", "summer_Precip_z"]:
    if ch in x_cols:
        k = x_cols.index(ch)
        show_field(X[idx, k], mask, f"{target_year} {ch}")

for ch in ["late_summer_EVI_z", "late_summer_LAI_z"]:
    if ch in y_cols:
        k = y_cols.index(ch)
        show_field(Y[idx, k], mask, f"{target_year} {ch}")

In [ ]:
# ============================================================
# Inspect yearly_grid_year_panels_vegpatched_v2 folder
# ============================================================

import os
import glob
import pandas as pd
from pathlib import Path

BASE_DIR = "/content/drive/MyDrive/GEE_NH_0_65_VegAgri_Yearly"
RAW_LIGHT_DIR = "/content/drive/MyDrive/GEE_NH_0_65_LightRaw_TargetRegions"

ANALYSIS_DIR = os.path.join(BASE_DIR, "_analysis_grid_year")
TABLE_DIR = os.path.join(ANALYSIS_DIR, "tables")

V2_YEAR_PANEL_DIR = os.path.join(TABLE_DIR, "yearly_grid_year_panels_vegpatched_v2")

print("V2_YEAR_PANEL_DIR:", V2_YEAR_PANEL_DIR)
print("Exists:", os.path.exists(V2_YEAR_PANEL_DIR))

all_files = sorted(glob.glob(os.path.join(V2_YEAR_PANEL_DIR, "**", "*"), recursive=True))
all_files = [f for f in all_files if os.path.isfile(f)]

print("Total files:", len(all_files))

ext_counts = {}
for f in all_files:
    ext = Path(f).suffix.lower()
    ext_counts[ext] = ext_counts.get(ext, 0) + 1

print("Extension counts:", ext_counts)

file_info = []
for f in all_files:
    file_info.append({
        "file": os.path.basename(f),
        "path": f,
        "ext": Path(f).suffix.lower(),
        "size_mb": os.path.getsize(f) / 1024 / 1024
    })

file_df = pd.DataFrame(file_info).sort_values(["ext", "file"])
display(file_df.head(50))

out_inventory = os.path.join(TABLE_DIR, "inventory_yearly_grid_year_panels_vegpatched_v2.csv")
file_df.to_csv(out_inventory, index=False)
print("Saved inventory:", out_inventory)

In [ ]:
# ============================================================
# Inspect parquet metadata/schema without loading full files
# ============================================================

import os
import glob
import re
import pandas as pd
import pyarrow.parquet as pq

BASE_DIR = "/content/drive/MyDrive/GEE_NH_0_65_VegAgri_Yearly"
ANALYSIS_DIR = os.path.join(BASE_DIR, "_analysis_grid_year")
TABLE_DIR = os.path.join(ANALYSIS_DIR, "tables")
V2_YEAR_PANEL_DIR = os.path.join(TABLE_DIR, "yearly_grid_year_panels_vegpatched_v2")

OUT_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset"
os.makedirs(OUT_DIR, exist_ok=True)

files = sorted(glob.glob(os.path.join(V2_YEAR_PANEL_DIR, "grid_year_y*_vegpatched.parquet")))
print("Files:", len(files))

meta_rows = []
all_cols_by_file = {}

for fp in files:
    year_match = re.search(r"y(20\d{2})", os.path.basename(fp))
    year = int(year_match.group(1)) if year_match else None

    pf = pq.ParquetFile(fp)
    schema = pf.schema_arrow
    cols = schema.names
    all_cols_by_file[os.path.basename(fp)] = cols

    meta_rows.append({
        "year": year,
        "file": os.path.basename(fp),
        "path": fp,
        "size_mb": os.path.getsize(fp) / 1024 / 1024,
        "n_rows": pf.metadata.num_rows,
        "n_row_groups": pf.num_row_groups,
        "n_cols": len(cols),
        "columns": "|".join(cols),
    })

    print(f"[OK] {year} | rows={pf.metadata.num_rows:,} | row_groups={pf.num_row_groups} | cols={len(cols)}")

meta_df = pd.DataFrame(meta_rows).sort_values("year")
display(meta_df[["year", "file", "size_mb", "n_rows", "n_row_groups", "n_cols"]])

meta_out = os.path.join(OUT_DIR, "v2_parquet_metadata_summary.csv")
meta_df.to_csv(meta_out, index=False)
print("Saved:", meta_out)

In [ ]:
# ============================================================
# Preview selected columns from one parquet file
# ============================================================

import os
import re
import pandas as pd
import pyarrow.parquet as pq

target_year = 2020
target_fp = [f for f in files if f"y{target_year}" in os.path.basename(f)][0]

pf = pq.ParquetFile(target_fp)
cols = pf.schema_arrow.names
lower_map = {c.lower(): c for c in cols}

# Candidate columns to inspect
must_try = [
    "grid_id", "lon", "lat", "region_id", "year",
    "row", "col", "grid_row", "grid_col",
]

# Add keyword hits, but keep manageable
kw_list = [
    "evi", "lai", "vpd", "lst", "sm_root", "soil", "precip", "tair", "temp",
    "elevation", "slope", "lc_type1", "landcover",
    "light", "ntl", "greenup", "peak", "dormancy"
]

selected = []
for c in must_try:
    if c in cols:
        selected.append(c)

for c in cols:
    cl = c.lower()
    if any(kw in cl for kw in kw_list):
        selected.append(c)

# de-duplicate
selected = list(dict.fromkeys(selected))

print("Selected columns:", len(selected))
for c in selected:
    print(" ", c)

# Read only selected columns
df_preview = pd.read_parquet(target_fp, columns=selected)

print("\nPreview shape:", df_preview.shape)
display(df_preview.head())
print("\nDtypes:")
display(df_preview.dtypes.reset_index().rename(columns={"index": "column", 0: "dtype"}))

# Basic unique stats
for c in ["year", "region_id", "grid_id", "lon", "lat"]:
    if c in df_preview.columns:
        print(f"{c}: n_unique={df_preview[c].nunique()}, sample={df_preview[c].dropna().unique()[:10]}")

In [ ]:
# ============================================================
# Step 1. Inspect vegetation patch CSV files
# ============================================================

import os
import re
import glob
import pandas as pd
from pathlib import Path

PATCH_CSV_DIR = "/content/drive/MyDrive/GEE_NH_0_65_VegAgri_Yearly"

patch_files = sorted(glob.glob(os.path.join(PATCH_CSV_DIR, "NH0_65_vegpatch_y*_R_*.csv")))

print("Patch CSV files found:", len(patch_files))

rows = []
for fp in patch_files:
    name = os.path.basename(fp)
    m = re.search(r"vegpatch_y(20\d{2})_(R_\d+_\d+)", name)
    year = int(m.group(1)) if m else None
    region = m.group(2) if m else None

    rows.append({
        "file": name,
        "path": fp,
        "year": year,
        "region_id": region,
        "size_mb": os.path.getsize(fp) / 1024 / 1024
    })

inv = pd.DataFrame(rows).sort_values(["year", "region_id"])
display(inv.head(30))
display(inv.groupby("year").agg(n_files=("file", "count"), size_mb=("size_mb", "sum")).reset_index())

OUT_INV = os.path.join(PATCH_CSV_DIR, "_inventory_vegpatch_csv_files.csv")
inv.to_csv(OUT_INV, index=False)
print("Saved inventory:", OUT_INV)

In [ ]:
# ============================================================
# Step 2. Preview one vegetation patch CSV
# ============================================================

import pandas as pd
import os

if len(patch_files) == 0:
    raise FileNotFoundError("No NH0_65_vegpatch_y*_R_*.csv files found.")

sample_fp = patch_files[0]
print("Sample file:", sample_fp)

preview = pd.read_csv(sample_fp, nrows=5)
print("Shape:", preview.shape)
print("Columns:")
print(preview.columns.tolist())
display(preview)

In [ ]:
# ============================================================
# Step 3. Convert vegetation patch CSVs to China-only yearly parquet
# ============================================================

import os
import re
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
from tqdm import tqdm

PATCH_CSV_DIR = "/content/drive/MyDrive/GEE_NH_0_65_VegAgri_Yearly"
CHINA_BOUNDARY_PATH = "/content/drive/MyDrive/China_boundary/china_boundary_full.gpkg"

OUT_PATCH_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched/_veg_patch_china_parquet"
os.makedirs(OUT_PATCH_DIR, exist_ok=True)

YEARS_TO_USE = list(range(2018, 2024))  # current seasonal dataset years

PATCH_COLS = [
    "grid_id", "lon", "lat", "region_id", "year", "step16",
    "date_start", "date_end", "doy_start", "doy_end",
    "EVI_patch", "NDVI_patch", "LAI_patch", "FPAR_patch",
    "MOD13_SummaryQA", "MOD15_FparLai_QC"
]

patch_files = sorted(glob.glob(os.path.join(PATCH_CSV_DIR, "NH0_65_vegpatch_y*_R_*.csv")))

def parse_year_region(fp):
    name = os.path.basename(fp)
    m = re.search(r"vegpatch_y(20\d{2})_(R_\d+_\d+)", name)
    if not m:
        return None, None
    return int(m.group(1)), m.group(2)

patch_files_use = []
for fp in patch_files:
    year, region = parse_year_region(fp)
    if year in YEARS_TO_USE:
        patch_files_use.append(fp)

print("Patch files for selected years:", len(patch_files_use))
display(pd.DataFrame([{
    "file": os.path.basename(f),
    "year": parse_year_region(f)[0],
    "region_id": parse_year_region(f)[1],
    "size_mb": os.path.getsize(f) / 1024 / 1024
} for f in patch_files_use]).head())

# Load China boundary
try:
    china = gpd.read_file(CHINA_BOUNDARY_PATH, layer="china_full").to_crs("EPSG:4326")
except Exception:
    china = gpd.read_file(CHINA_BOUNDARY_PATH).to_crs("EPSG:4326")

china = china.dissolve().reset_index(drop=True)

# Small buffer helps retain small territories at 0.1° grid
china_filter = china.copy()
china_filter["geometry"] = china_filter.geometry.buffer(0.03)

def clean_patch_chunk(df):
    # Keep known columns only
    keep = [c for c in PATCH_COLS if c in df.columns]
    df = df[keep].copy()

    # numeric conversion
    for c in ["lon", "lat", "year", "step16", "doy_start", "doy_end",
              "EVI_patch", "NDVI_patch", "LAI_patch", "FPAR_patch",
              "MOD13_SummaryQA", "MOD15_FparLai_QC"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # GEE unmask value
    for c in ["EVI_patch", "NDVI_patch", "LAI_patch", "FPAR_patch"]:
        if c in df.columns:
            df.loc[df[c] <= -9990, c] = np.nan

    # Conservative valid ranges
    if "EVI_patch" in df.columns:
        df.loc[(df["EVI_patch"] < -0.2) | (df["EVI_patch"] > 1.2), "EVI_patch"] = np.nan

    if "NDVI_patch" in df.columns:
        df.loc[(df["NDVI_patch"] < -0.2) | (df["NDVI_patch"] > 1.2), "NDVI_patch"] = np.nan

    if "LAI_patch" in df.columns:
        df.loc[(df["LAI_patch"] < 0) | (df["LAI_patch"] > 12), "LAI_patch"] = np.nan

    if "FPAR_patch" in df.columns:
        df.loc[(df["FPAR_patch"] < 0) | (df["FPAR_patch"] > 1.2), "FPAR_patch"] = np.nan

    return df

def china_filter_points(df):
    if len(df) == 0:
        return df

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["lon"], df["lat"]),
        crs="EPSG:4326"
    )

    gdf2 = gpd.sjoin(
        gdf,
        china_filter[["geometry"]],
        how="inner",
        predicate="within"
    )

    drop_cols = [c for c in ["geometry", "index_right"] if c in gdf2.columns]
    out = pd.DataFrame(gdf2.drop(columns=drop_cols))
    return out

# Process year by year
summary_rows = []

for year in YEARS_TO_USE:
    year_files = [fp for fp in patch_files_use if parse_year_region(fp)[0] == year]
    print(f"\n=== Year {year}: {len(year_files)} patch files ===")

    parts = []

    for fp in tqdm(year_files):
        try:
            # chunk reading for large CSVs
            for chunk in pd.read_csv(fp, usecols=lambda c: c in PATCH_COLS, chunksize=300_000):
                chunk = clean_patch_chunk(chunk)

                if len(chunk) == 0:
                    continue

                # bbox prefilter before geopandas
                chunk = chunk[
                    (chunk["lon"] >= 70) & (chunk["lon"] <= 138) &
                    (chunk["lat"] >= 15) & (chunk["lat"] <= 56)
                ].copy()

                if len(chunk) == 0:
                    continue

                chunk = china_filter_points(chunk)

                if len(chunk) > 0:
                    parts.append(chunk)

        except pd.errors.EmptyDataError:
            print("[EMPTY]", os.path.basename(fp))
        except Exception as e:
            print("[ERROR]", os.path.basename(fp), repr(e))

    if len(parts) == 0:
        print(f"[WARN] No China patch rows for {year}")
        continue

    year_df = pd.concat(parts, ignore_index=True)

    # Drop duplicates by averaging patch variables
    key_cols = ["grid_id", "year", "step16"]
    meta_cols = ["grid_id", "lon", "lat", "region_id", "year", "step16", "date_start", "date_end", "doy_start", "doy_end"]

    value_cols = [c for c in ["EVI_patch", "NDVI_patch", "LAI_patch", "FPAR_patch", "MOD13_SummaryQA", "MOD15_FparLai_QC"] if c in year_df.columns]

    meta = (
        year_df[meta_cols]
        .drop_duplicates(subset=key_cols)
    )

    val = (
        year_df
        .groupby(key_cols, as_index=False)[value_cols]
        .mean()
    )

    year_df = meta.merge(val, on=key_cols, how="left")

    out_fp = os.path.join(OUT_PATCH_DIR, f"china_vegpatch_y{year}.parquet")
    year_df.to_parquet(out_fp, index=False)

    row = {
        "year": year,
        "rows": len(year_df),
        "grids": year_df["grid_id"].nunique(),
        "steps": year_df["step16"].nunique(),
        "EVI_patch_valid": year_df["EVI_patch"].notna().mean() if "EVI_patch" in year_df.columns else np.nan,
        "LAI_patch_valid": year_df["LAI_patch"].notna().mean() if "LAI_patch" in year_df.columns else np.nan,
        "out_fp": out_fp
    }
    summary_rows.append(row)

    print("[OK]", row)

summary = pd.DataFrame(summary_rows)
summary_fp = os.path.join(OUT_PATCH_DIR, "china_vegpatch_year_summary.csv")
summary.to_csv(summary_fp, index=False)

print("\nSaved patch summary:", summary_fp)
display(summary)

In [ ]:
# ============================================================
# Step 4. Patch EVI/LAI in China dynamic boundary_full parquet
# ============================================================

import os
import glob
import re
import numpy as np
import pandas as pd

DYN_STATIC_DIR = "/content/drive/MyDrive/GEE_China_VegClimate_NO_boundary_full"
PATCH_PARQUET_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched/_veg_patch_china_parquet"

OUT_PATCHED_DIR = "/content/drive/MyDrive/GEE_China_VegClimate_NO_boundary_full_vegpatched"
os.makedirs(OUT_PATCHED_DIR, exist_ok=True)

YEARS_TO_USE = list(range(2018, 2024))

REPLACE_EXISTING = False  # False = only fill missing EVI/LAI; True = overwrite with patch when patch valid

def find_dynamic_file(year):
    candidates = sorted(glob.glob(os.path.join(DYN_STATIC_DIR, f"*dynamic*y{year}*boundary_full*.parquet")))
    if len(candidates) == 0:
        candidates = sorted(glob.glob(os.path.join(DYN_STATIC_DIR, f"*dynamic*y{year}*.parquet")))
    return candidates[0] if candidates else None

def find_static_file(year):
    candidates = sorted(glob.glob(os.path.join(DYN_STATIC_DIR, f"*static*y{year}*boundary_full*.parquet")))
    if len(candidates) == 0:
        candidates = sorted(glob.glob(os.path.join(DYN_STATIC_DIR, f"*static*y{year}*.parquet")))
    return candidates[0] if candidates else None

summary_rows = []

for year in YEARS_TO_USE:
    dyn_fp = find_dynamic_file(year)
    patch_fp = os.path.join(PATCH_PARQUET_DIR, f"china_vegpatch_y{year}.parquet")

    print("\n" + "="*90)
    print("Year:", year)
    print("Dynamic:", dyn_fp)
    print("Patch:", patch_fp)

    if dyn_fp is None:
        print("[WARN] dynamic file not found; skip")
        continue

    if not os.path.exists(patch_fp):
        print("[WARN] patch file not found; copy original dynamic without patch")
        dyn = pd.read_parquet(dyn_fp)
        out_fp = os.path.join(OUT_PATCHED_DIR, f"china_dynamic_y{year}_boundary_full_vegpatched.parquet")
        dyn.to_parquet(out_fp, index=False)
        continue

    dyn = pd.read_parquet(dyn_fp)
    patch = pd.read_parquet(patch_fp)

    # Ensure key types
    for df in [dyn, patch]:
        df["year"] = pd.to_numeric(df["year"], errors="coerce").astype("Int64")
        df["step16"] = pd.to_numeric(df["step16"], errors="coerce").astype("Int64")

    key_cols = ["grid_id", "year", "step16"]

    # Keep only patch columns needed
    patch_keep = key_cols + [c for c in ["EVI_patch", "LAI_patch", "NDVI_patch", "FPAR_patch"] if c in patch.columns]
    patch = patch[patch_keep].copy()

    # Drop duplicate patch rows by mean
    value_cols = [c for c in ["EVI_patch", "LAI_patch", "NDVI_patch", "FPAR_patch"] if c in patch.columns]
    patch = patch.groupby(key_cols, as_index=False)[value_cols].mean()

    # Diagnostics before merge
    dyn_rows = len(dyn)
    dyn_grids = dyn["grid_id"].nunique()
    dyn_evi_before = dyn["EVI"].notna().mean() if "EVI" in dyn.columns else np.nan
    dyn_lai_before = dyn["LAI"].notna().mean() if "LAI" in dyn.columns else np.nan

    # Merge patch onto dynamic
    merged = dyn.merge(
        patch,
        on=key_cols,
        how="left",
        indicator=True,
        validate="many_to_one"
    )

    matched_frac = (merged["_merge"] == "both").mean()

    # Add NDVI/FPAR patch columns as auxiliary if not already present
    if "NDVI_patch" in merged.columns and "NDVI_patch_aux" not in merged.columns:
        merged["NDVI_patch_aux"] = merged["NDVI_patch"]

    if "FPAR_patch" in merged.columns and "FPAR_patch_aux" not in merged.columns:
        merged["FPAR_patch_aux"] = merged["FPAR_patch"]

    # Patch EVI
    if "EVI" in merged.columns and "EVI_patch" in merged.columns:
        if REPLACE_EXISTING:
            use = merged["EVI_patch"].notna()
        else:
            use = merged["EVI"].isna() & merged["EVI_patch"].notna()

        merged["EVI_from_patch"] = use.astype("int8")
        merged.loc[use, "EVI"] = merged.loc[use, "EVI_patch"]
    else:
        merged["EVI_from_patch"] = 0

    # Patch LAI
    if "LAI" in merged.columns and "LAI_patch" in merged.columns:
        if REPLACE_EXISTING:
            use = merged["LAI_patch"].notna()
        else:
            use = merged["LAI"].isna() & merged["LAI_patch"].notna()

        merged["LAI_from_patch"] = use.astype("int8")
        merged.loc[use, "LAI"] = merged.loc[use, "LAI_patch"]
    else:
        merged["LAI_from_patch"] = 0

    # Save unmatched patch rows for diagnostics
    dyn_keys = dyn[key_cols].drop_duplicates()
    patch_unmatched = patch.merge(dyn_keys, on=key_cols, how="left", indicator=True)
    patch_unmatched = patch_unmatched[patch_unmatched["_merge"] == "left_only"].drop(columns=["_merge"])

    unmatched_fp = os.path.join(OUT_PATCHED_DIR, f"patch_unmatched_y{year}.parquet")
    patch_unmatched.to_parquet(unmatched_fp, index=False)

    # Drop merge helper and raw patch columns, but keep flags and aux if desired
    drop_cols = ["_merge", "EVI_patch", "LAI_patch", "NDVI_patch", "FPAR_patch"]
    merged = merged.drop(columns=[c for c in drop_cols if c in merged.columns])

    out_fp = os.path.join(OUT_PATCHED_DIR, f"china_dynamic_y{year}_boundary_full_vegpatched.parquet")
    merged.to_parquet(out_fp, index=False)

    row = {
        "year": year,
        "rows": len(merged),
        "grids": merged["grid_id"].nunique(),
        "steps": merged["step16"].nunique(),
        "EVI_valid_before": dyn_evi_before,
        "EVI_valid_after": merged["EVI"].notna().mean() if "EVI" in merged.columns else np.nan,
        "LAI_valid_before": dyn_lai_before,
        "LAI_valid_after": merged["LAI"].notna().mean() if "LAI" in merged.columns else np.nan,
        "EVI_filled_rows": int(merged["EVI_from_patch"].sum()),
        "LAI_filled_rows": int(merged["LAI_from_patch"].sum()),
        "patch_match_frac_on_dynamic_rows": matched_frac,
        "patch_unmatched_rows": len(patch_unmatched),
        "out_fp": out_fp,
        "unmatched_fp": unmatched_fp
    }
    summary_rows.append(row)

    print("[OK] patched dynamic saved:", out_fp)
    print(row)

summary = pd.DataFrame(summary_rows)
summary_fp = os.path.join(OUT_PATCHED_DIR, "vegpatch_merge_summary.csv")
summary.to_csv(summary_fp, index=False)

print("\nSaved patch merge summary:", summary_fp)
display(summary)

In [ ]:
# ============================================================
# Step 5. Copy static files into vegpatched folder
# ============================================================

import os
import glob
import shutil

DYN_STATIC_DIR = "/content/drive/MyDrive/GEE_China_VegClimate_NO_boundary_full"
OUT_PATCHED_DIR = "/content/drive/MyDrive/GEE_China_VegClimate_NO_boundary_full_vegpatched"

os.makedirs(OUT_PATCHED_DIR, exist_ok=True)

static_files = sorted(glob.glob(os.path.join(DYN_STATIC_DIR, "*static*y*boundary_full*.parquet")))
if len(static_files) == 0:
    static_files = sorted(glob.glob(os.path.join(DYN_STATIC_DIR, "*static*y*.parquet")))

print("Static files:", len(static_files))

for fp in static_files:
    name = os.path.basename(fp)

    # standardize output name
    # example: china_static_y2018_boundary_full.parquet
    out_name = name
    if "vegpatched" not in out_name:
        out_name = out_name.replace(".parquet", "_vegpatched.parquet")

    out_fp = os.path.join(OUT_PATCHED_DIR, out_name)
    shutil.copy2(fp, out_fp)
    print("[COPIED]", os.path.basename(fp), "->", os.path.basename(out_fp))

In [ ]:
# ============================================================
# Step 6. Build patched China 16-day core panel
# ============================================================

import os
import glob
import re
import numpy as np
import pandas as pd

IN_DIR = "/content/drive/MyDrive/GEE_China_VegClimate_NO_boundary_full_vegpatched"
OUT_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
os.makedirs(OUT_DIR, exist_ok=True)

dyn_files = sorted(glob.glob(os.path.join(IN_DIR, "*dynamic*y*vegpatched*.parquet")))
sta_files = sorted(glob.glob(os.path.join(IN_DIR, "*static*y*vegpatched*.parquet")))

print("Dynamic files:", len(dyn_files))
print("Static files:", len(sta_files))

if len(dyn_files) == 0:
    raise FileNotFoundError(f"No patched dynamic parquet found in {IN_DIR}")

if len(sta_files) == 0:
    raise FileNotFoundError(f"No static parquet found in {IN_DIR}")

def extract_year_from_name(fp):
    m = re.search(r"y(20\d{2})", os.path.basename(fp))
    return int(m.group(1)) if m else None

dyn_cols = [
    "grid_id", "lon", "lat", "region_id",
    "year", "step16", "date_start", "date_end", "doy_start", "doy_end",
    "EVI", "LAI",
    "VPD", "LST_Day", "SM_rootproxy", "Precip", "Tair",
    "EVI_from_patch", "LAI_from_patch",
    "NDVI_patch_aux", "FPAR_patch_aux"
]

sta_cols = [
    "grid_id", "lon", "lat", "region_id",
    "year_static",
    "elevation", "slope", "LC_Type1",
    "lc_veg_frac", "evi_peak_frac", "evi_amp_frac", "valid_frac",
    "Greenup_1", "Peak_1", "Dormancy_1"
]

def read_keep(fp, cols):
    df = pd.read_parquet(fp)
    keep = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        print(f"[WARN] {os.path.basename(fp)} missing columns: {missing}")
    return df[keep].copy()

# Load dynamic
dyn_parts = []
for fp in dyn_files:
    year = extract_year_from_name(fp)
    df = read_keep(fp, dyn_cols)

    if "year" not in df.columns and year is not None:
        df["year"] = year

    subset = [c for c in ["grid_id", "year", "step16"] if c in df.columns]
    if subset:
        df = df.drop_duplicates(subset=subset)

    dyn_parts.append(df)

    print(
        f"[DYN] {os.path.basename(fp)} rows={len(df):,}, "
        f"grids={df['grid_id'].nunique():,}, "
        f"EVI_valid={df['EVI'].notna().mean():.3f}, "
        f"LAI_valid={df['LAI'].notna().mean():.3f}"
    )

dynamic = pd.concat(dyn_parts, ignore_index=True)

print("\nDynamic total:", dynamic.shape)
print("Years:", sorted(dynamic["year"].dropna().unique()))
print("Grids:", dynamic["grid_id"].nunique())

# Load static
sta_parts = []
for fp in sta_files:
    year = extract_year_from_name(fp)
    df = read_keep(fp, sta_cols)

    if "year_static" not in df.columns and year is not None:
        df["year_static"] = year

    subset = [c for c in ["grid_id", "year_static"] if c in df.columns]
    if subset:
        df = df.drop_duplicates(subset=subset)

    sta_parts.append(df)

    print(f"[STA] {os.path.basename(fp)} rows={len(df):,}, grids={df['grid_id'].nunique():,}")

static = pd.concat(sta_parts, ignore_index=True)

print("\nStatic total:", static.shape)
print("Years:", sorted(static["year_static"].dropna().unique()))
print("Grids:", static["grid_id"].nunique())

# Merge
static_ren = static.rename(columns={"year_static": "year"})
drop_from_static = [c for c in ["lon", "lat", "region_id"] if c in static_ren.columns]
static_ren = static_ren.drop(columns=drop_from_static)

panel = dynamic.merge(
    static_ren,
    on=["grid_id", "year"],
    how="left",
    validate="many_to_one"
)

# numeric
num_cols = [
    "lon", "lat", "year", "step16", "doy_start", "doy_end",
    "EVI", "LAI", "VPD", "LST_Day", "SM_rootproxy", "Precip", "Tair",
    "elevation", "slope", "LC_Type1"
]
for c in num_cols:
    if c in panel.columns:
        panel[c] = pd.to_numeric(panel[c], errors="coerce")

panel["doy_mid"] = (panel["doy_start"] + panel["doy_end"]) / 2.0

out_fp = os.path.join(OUT_DIR, "china_panel_16day_core_patched.parquet")
panel.to_parquet(out_fp, index=False)

print("\n[OK] Saved patched core panel:")
print(out_fp)
print("Panel:", panel.shape)
print("EVI valid:", panel["EVI"].notna().mean())
print("LAI valid:", panel["LAI"].notna().mean())
print("EVI_from_patch rows:", int(panel["EVI_from_patch"].sum()) if "EVI_from_patch" in panel.columns else 0)
print("LAI_from_patch rows:", int(panel["LAI_from_patch"].sum()) if "LAI_from_patch" in panel.columns else 0)
display(panel.head())

In [ ]:
# ============================================================
# Step 7. Compute anomalies for patched 16-day panel
# ============================================================

import os
import numpy as np
import pandas as pd

OUT_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
in_fp = os.path.join(OUT_DIR, "china_panel_16day_core_patched.parquet")

panel = pd.read_parquet(in_fp)

anom_vars = [
    "EVI", "LAI",
    "VPD", "LST_Day", "SM_rootproxy", "Precip", "Tair"
]

for c in anom_vars:
    panel[c] = pd.to_numeric(panel[c], errors="coerce")

# Conservative cleaning
panel.loc[(panel["EVI"] < -0.2) | (panel["EVI"] > 1.2), "EVI"] = np.nan
panel.loc[(panel["LAI"] < 0) | (panel["LAI"] > 12), "LAI"] = np.nan

# Build climatology
clim = panel.groupby(["grid_id", "step16"])[anom_vars].agg(["mean", "std", "count"])
clim.columns = [f"{var}_clim_{stat}" for var, stat in clim.columns]
clim = clim.reset_index()

panel2 = panel.merge(
    clim,
    on=["grid_id", "step16"],
    how="left",
    validate="many_to_one"
)

for v in anom_vars:
    mean_col = f"{v}_clim_mean"
    std_col = f"{v}_clim_std"

    panel2[f"{v}_anom"] = panel2[v] - panel2[mean_col]

    std = panel2[std_col].replace(0, np.nan)
    std = std.where(std > 1e-6, np.nan)

    panel2[f"{v}_z"] = panel2[f"{v}_anom"] / std
    panel2[f"{v}_z"] = panel2[f"{v}_z"].clip(-5, 5)

clim_fp = os.path.join(OUT_DIR, "china_16day_grid_climatology_patched.parquet")
anom_fp = os.path.join(OUT_DIR, "china_panel_16day_anomaly_patched.parquet")

clim.to_parquet(clim_fp, index=False)
panel2.to_parquet(anom_fp, index=False)

print("[OK] Saved climatology:", clim_fp)
print("[OK] Saved anomaly panel:", anom_fp)
print("Panel shape:", panel2.shape)

summary = []
for v in anom_vars:
    summary.append({
        "var": v,
        "raw_valid": panel2[v].notna().mean(),
        "anom_valid": panel2[f"{v}_anom"].notna().mean(),
        "z_valid": panel2[f"{v}_z"].notna().mean(),
        "mean_clim_count": panel2[f"{v}_clim_count"].mean(),
        "median_clim_count": panel2[f"{v}_clim_count"].median(),
        "z_mean": panel2[f"{v}_z"].mean(),
        "z_std": panel2[f"{v}_z"].std()
    })

summary = pd.DataFrame(summary)
display(summary)
summary.to_csv(os.path.join(OUT_DIR, "anomaly_variable_summary_patched.csv"), index=False)

In [ ]:
# ============================================================
# Step 8. Build patched seasonal response panel
# ============================================================

import os
import numpy as np
import pandas as pd

OUT_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
in_fp = os.path.join(OUT_DIR, "china_panel_16day_anomaly_patched.parquet")

df = pd.read_parquet(in_fp)

df["doy_mid"] = (
    pd.to_numeric(df["doy_start"], errors="coerce") +
    pd.to_numeric(df["doy_end"], errors="coerce")
) / 2.0

season_defs = {
    "spring": (60, 151),
    "summer": (152, 243),
    "late_summer": (213, 273),
    "autumn": (244, 304),
}

z_vars = [
    "EVI_z", "LAI_z",
    "VPD_z", "LST_Day_z", "SM_rootproxy_z", "Precip_z", "Tair_z"
]

anom_vars = [
    "EVI_anom", "LAI_anom",
    "VPD_anom", "LST_Day_anom", "SM_rootproxy_anom", "Precip_anom", "Tair_anom"
]

raw_vars = [
    "EVI", "LAI",
    "VPD", "LST_Day", "SM_rootproxy", "Precip", "Tair"
]

flag_vars = [
    "EVI_from_patch", "LAI_from_patch"
]

static_vars = [
    "elevation", "slope", "LC_Type1",
    "lc_veg_frac", "evi_peak_frac", "evi_amp_frac", "valid_frac",
    "Greenup_1", "Peak_1", "Dormancy_1"
]

season_parts = []

for season, (d0, d1) in season_defs.items():
    sub = df[(df["doy_mid"] >= d0) & (df["doy_mid"] <= d1)].copy()

    agg_vars = [v for v in z_vars + anom_vars + raw_vars if v in sub.columns]

    agg = sub.groupby(["grid_id", "year"], as_index=False)[agg_vars].mean()

    count = (
        sub
        .groupby(["grid_id", "year"], as_index=False)
        .size()
        .rename(columns={"size": f"{season}_n_steps"})
    )

    agg = agg.merge(count, on=["grid_id", "year"], how="left")

    # Patch flags: fraction of seasonal records filled from patch
    for flag in flag_vars:
        if flag in sub.columns:
            flag_agg = (
                sub
                .groupby(["grid_id", "year"], as_index=False)[flag]
                .mean()
                .rename(columns={flag: f"{season}_{flag}_frac"})
            )
            agg = agg.merge(flag_agg, on=["grid_id", "year"], how="left")

    rename = {v: f"{season}_{v}" for v in agg_vars}
    agg = agg.rename(columns=rename)

    season_parts.append(agg)

seasonal = season_parts[0]
for part in season_parts[1:]:
    seasonal = seasonal.merge(part, on=["grid_id", "year"], how="outer")

static_cols = ["grid_id", "year", "lon", "lat", "region_id"] + [v for v in static_vars if v in df.columns]
static = df[static_cols].drop_duplicates(subset=["grid_id", "year"])
seasonal = seasonal.merge(static, on=["grid_id", "year"], how="left")

x_cols = [
    "spring_EVI_z",
    "spring_LAI_z",
    "summer_VPD_z",
    "summer_LST_Day_z",
    "summer_SM_rootproxy_z",
    "summer_Precip_z",
    "summer_Tair_z",
    "elevation",
    "slope",
    "LC_Type1"
]

y_cols = [
    "late_summer_EVI_z",
    "late_summer_LAI_z"
]

required_x = [c for c in x_cols if c in seasonal.columns]

seasonal["valid_x"] = seasonal[required_x].notna().all(axis=1)
seasonal["valid_y_evi"] = seasonal["late_summer_EVI_z"].notna()
seasonal["valid_y_lai"] = seasonal["late_summer_LAI_z"].notna()
seasonal["valid_model_any_y"] = seasonal["valid_x"] & (seasonal["valid_y_evi"] | seasonal["valid_y_lai"])
seasonal["valid_model_both_y"] = seasonal["valid_x"] & seasonal["valid_y_evi"] & seasonal["valid_y_lai"]

out_fp = os.path.join(OUT_DIR, "china_seasonal_response_panel_patched.parquet")
seasonal.to_parquet(out_fp, index=False)

print("[OK] Saved patched seasonal response panel:", out_fp)
print("Rows:", len(seasonal))
print("Years:", sorted(seasonal["year"].dropna().unique()))
print("Grids:", seasonal["grid_id"].nunique())
print("Valid X:", seasonal["valid_x"].sum())
print("Valid X + any Y:", seasonal["valid_model_any_y"].sum())
print("Valid X + both Y:", seasonal["valid_model_both_y"].sum())

comp = []
for c in x_cols + y_cols + ["late_summer_EVI_anom", "late_summer_LAI_anom"]:
    if c in seasonal.columns:
        comp.append({"column": c, "valid_frac": seasonal[c].notna().mean()})

comp = pd.DataFrame(comp)
display(comp)
comp.to_csv(os.path.join(OUT_DIR, "seasonal_response_completeness_patched.csv"), index=False)

display(seasonal.head())

In [ ]:
# ============================================================
# Step 9. Compare before vs after patching
# ============================================================

import os
import pandas as pd

OLD_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset"
NEW_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"

old_fp = os.path.join(OLD_DIR, "china_seasonal_response_panel.parquet")
new_fp = os.path.join(NEW_DIR, "china_seasonal_response_panel_patched.parquet")

old = pd.read_parquet(old_fp)
new = pd.read_parquet(new_fp)

cols_check = [
    "spring_EVI_z", "spring_LAI_z",
    "late_summer_EVI_z", "late_summer_LAI_z",
    "valid_x", "valid_model_both_y"
]

rows = []
for c in cols_check:
    if c in old.columns and c in new.columns:
        if old[c].dtype == bool:
            old_val = old[c].mean()
            new_val = new[c].mean()
        else:
            old_val = old[c].notna().mean()
            new_val = new[c].notna().mean()

        rows.append({
            "metric": c,
            "before": old_val,
            "after": new_val,
            "improvement": new_val - old_val
        })

cmp = pd.DataFrame(rows)
display(cmp)

cmp.to_csv(os.path.join(NEW_DIR, "patch_before_after_comparison.csv"), index=False)

# Year-level valid counts
def year_valid_table(df, label):
    out = (
        df.groupby("year")
        .agg(
            rows=("grid_id", "count"),
            grids=("grid_id", "nunique"),
            valid_x=("valid_x", "sum"),
            valid_both_y=("valid_model_both_y", "sum")
        )
        .reset_index()
    )
    out["dataset"] = label
    return out

year_cmp = pd.concat([
    year_valid_table(old, "before"),
    year_valid_table(new, "after")
], ignore_index=True)

display(year_cmp)
year_cmp.to_csv(os.path.join(NEW_DIR, "patch_before_after_year_counts.csv"), index=False)

In [ ]:
OUT_DIR = "/content/drive/MyDrive/China_VegClimate_ResponseOperator_Dataset_patched"
in_fp = os.path.join(OUT_DIR, "china_seasonal_response_panel_patched.parquet")